# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahid-Imran/flyrank-ml-internship-mahid-assignment2/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1:

The FlyRank paper reports improved ranking performance compared with baseline approaches.

Methodology question:

How were the labels generated?
Were the ranking outcomes collected independently from the training data?

Why this matters:

A validation design that closely matches real deployment conditions is necessary to support the performance claim.

Finding 2:

The paper reports that selected features contributed strongly to prediction performance.

Methodology question:

Were these features available before prediction time, or could they contain future information?

Why this matters:

Features generated after the outcome may create leakage and overestimate model performance.

Methodology Question:

How were the labels created for the ranking/content outcomes?

I would verify whether the labels were generated using information available before the prediction period. If the label generation process uses future information, the reported performance may be optimistic.

Why this matters:

A reliable validation process requires that training information and evaluation outcomes represent realistic deployment conditions.

Methodology Question:

Does the validation design support the performance claim?

I would check whether the test data represents truly unseen examples and whether the evaluation split matches the real-world use case.

Why this matters:

A random split may overestimate performance when similar pages or historical patterns appear in both training and testing data.

In [1]:
import pandas as pd
import numpy as np


DATA_URL = (
    "https://raw.githubusercontent.com/"
    "flyrank-bih/flyrank-ml-internship-starter/"
    "main/data/raw/content_refresh_anonymized.csv"
)


df = pd.read_csv(DATA_URL)

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## Week-5 Model Re-evaluation

The Week-5 model used a random train-test split.

For this audit, I compare:

1. Original random split evaluation
2. Time-aware split evaluation

The time-aware split is more appropriate because the model objective is to identify possible future content performance decline.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

In [3]:
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "flyrank-bih/flyrank-ml-internship-starter/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
df["target"] = (
    df["trend_direction"]
    .eq("down")
    .astype(int)
)

df["target"].value_counts()

,count
target,
1,16262
0,13738


In [5]:
features = [
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "engagement_rate",
    "content_age_days"
]


X = df[features]

y = df["target"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


model_random = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)


model_random.fit(
    X_train,
    y_train
)


pred_random = model_random.predict(
    X_test
)


random_precision = precision_score(
    y_test,
    pred_random
)

random_recall = recall_score(
    y_test,
    pred_random
)

random_f1 = f1_score(
    y_test,
    pred_random
)


print("Random Split")
print("Precision:", random_precision)
print("Recall:", random_recall)
print("F1:", random_f1)

Random Split
Precision: 0.6889763779527559
Recall: 0.7533825338253383
F1: 0.7197414806110458


## Time-Aware Validation

The original random split mixes historical and future-like examples.

To simulate deployment conditions, the model is trained on earlier observations and evaluated on later observations.

In [7]:
df = df.sort_values(
    "days_since_last_update"
)


split_index = int(len(df)*0.8)


train_df = df.iloc[:split_index]

test_df = df.iloc[split_index:]


X_train_time = train_df[features]

y_train_time = train_df["target"]


X_test_time = test_df[features]

y_test_time = test_df["target"]

In [8]:
model_time = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)


model_time.fit(
    X_train_time,
    y_train_time
)


pred_time = model_time.predict(
    X_test_time
)


time_precision = precision_score(
    y_test_time,
    pred_time
)


time_recall = recall_score(
    y_test_time,
    pred_time
)


time_f1 = f1_score(
    y_test_time,
    pred_time
)


print("Time Aware Split")
print("Precision:", time_precision)
print("Recall:", time_recall)
print("F1:", time_f1)

Time Aware Split
Precision: 0.696289293311274
Recall: 0.8038199181446112
F1: 0.7462006079027356


## Before vs After Comparison


| Validation Method | Precision | Recall | F1 |
|---|---|---|---|
| Random Split | measured above | measured above | measured above |
| Time-Aware Split | measured above | measured above | measured above |


Observation:

The time-aware validation provides a more realistic estimate because it evaluates the model on later observations rather than randomly mixed examples.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Feature Leakage Audit

A feature leakage audit checks whether any feature contains information that would not be available at prediction time.

For each feature, I reviewed whether it represents historical information or whether it could contain future information.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.